In [10]:
!pip install lightfm-next

In [11]:
import pandas as pd
import numpy as np
import kagglehub
import os

# Download latest version
path = kagglehub.dataset_download("parasharmanas/movie-recommendation-system")

print("Path to dataset files:", path)

print(f'Downloaded files :')
for file in os.listdir(path):
    print(f"{file}")

Path to dataset files: /home/ashish-d/.cache/kagglehub/datasets/parasharmanas/movie-recommendation-system/versions/1
Downloaded files :
ratings.csv
movies.csv


In [12]:
from lightfm import LightFM
from lightfm.data import Dataset

movies = pd.read_csv(os.path.join(path , "movies.csv"))
ratings = pd.read_csv(os.path.join(path , "ratings.csv"))

movies["genre_list"] = movies["genres"].apply(lambda x : x.split("|"))

In [13]:
# creating unique genre list

genres = set([genre for sublist in movies["genre_list"] for genre in sublist])

In [14]:
# creating unique user ids

users = np.unique(ratings.userId)
cinemas = np.unique(movies.movieId)

In [15]:
dataset = Dataset()

dataset.fit(
    users = users,
    items = cinemas,
    item_features=genres
)

In [16]:
item_features_input = list(zip(movies.movieId , movies.genre_list))

interactions , weights = dataset.build_interactions(zip(ratings.userId , ratings.movieId , ratings.rating))

item_features = dataset.build_item_features(item_features_input)

In [17]:
model = LightFM(loss="warp", no_components=30)

print("Training the Hybrid LightFM model with genres...")
model.fit(
    interactions,
    item_features=item_features,
    epochs=20,
    num_threads=2,
    verbose = True
)
print("Training complete!")

user_id_map, _, item_id_map, _ = dataset.mapping()
user_index = user_id_map[1]
all_item_indexes = np.array(list(item_id_map.values()))

Training the Hybrid LightFM model with genres...


Epoch: 100%|██████████| 20/20 [12:10<00:00, 36.55s/it]

Training complete!


In [20]:
# Predict scores for all items for this user
scores = model.predict(user_index, all_item_indexes, item_features=item_features)
print("\nTop 5 Predicted ranking scores for User U1:")

#Make a copy so we don't destroy your original scores data
temp_scores = scores.copy()

print("\nTop 5 Recommendations using argmax loop:")

for rank in range(1, 6):
    # 1. Find the index of the current highest score
    max_idx = np.argmax(temp_scores)
    
    # 2. Retrieve the movie ID and score
    movie_id_str = list(item_id_map.keys())[max_idx]
    movie_score = temp_scores[max_idx]
    
    print(f"Rank {rank}: Movie ID {movie_id_str} | Score: {movie_score:.4f}")
    
    # 3. MASK IT: Set this maximum value to a very low number 
    # so argmax ignores it on the next loop iteration
    temp_scores[max_idx] = -np.inf


Top 5 Predicted ranking scores for User U1:

Top 5 Recommendations using argmax loop:
Rank 1: Movie ID 4973 | Score: 4.8863
Rank 2: Movie ID 2858 | Score: 4.4244
Rank 3: Movie ID 1230 | Score: 4.2534
Rank 4: Movie ID 750 | Score: 4.1802
Rank 5: Movie ID 7361 | Score: 4.1263
